# Code to deal with Prospector output files! Requires own environment

In [ ]:
import os
import glob
import numpy as np
import pickle as pkl
import pandas as pd
import prospect.io.read_results as reader
from prospect.utils.plotting import get_percentiles, get_best
from corner import quantile
import h5py


from astropy.io import fits
from astropy.table import Table
from astropy.cosmology import WMAP9 as cosmo
from prospect.models.transforms import logsfr_ratios_to_sfrs
from prospect.sources import FastStepBasis
from astropy.cosmology import Planck18 as cosmo
from prospect.models.sedmodel import PolySpecModel, SpecModel


# Prospector Loading Function

In [ ]:
def load_prospector_results(galaxy_id, prosp_dir):
    """Function to load Prospector result .h5 files and disect its 
    data structure to be used for easy plotting.

    Args:
        galaxy_id (int): The ID of the galaxy for which to load results
        prosp_dir (str): The directory containing the Prospector output files

    Returns:
        dict: A comprehensive dictionary storing the parameters samples, MAP values and quantiles
    """
    
    # Load the h5 file for the given galaxy ID
    h5_files = glob.glob(os.path.join(prosp_dir, f'*{galaxy_id}*.h5'))
    
    try:
        h5_file = h5_files[0]
        print(f"Loading Prospector output file: {h5_file}")
    except IndexError:
        print(f"No PROSPECTOR results found for objid {galaxy_id}.")
        return None

    # Load PROSPECTOR results
    results, obs, model = reader.results_from(h5_file)
        
    # Now we have to exclude the last 3 parameters from the fit
    map_parameters = get_best(results)
    
    # Extract labels for parameters that were "free" (fitted)
    labels = map_parameters[0]

    # Build the MAP dictionary
    MAP = {}
    for a,b in zip(map_parameters[0], map_parameters[1]):
        MAP[a] = b
    
    # Extract chains, weights and the MAP index
    chain = results['chain']
    weights = results['weights']
    imax = np.argmax(results['lnprobability'])
    
    data = {
            'meta': {'labels': map_parameters[0], 'map_idx': imax, 'weights': weights},
            'params': {}
        }

    perc = get_percentiles(results, [16, 50, 84])    
    
    for i, name in enumerate(map_parameters[0]):
            # Use the flattened chain for statistics
            param_samples = chain[:, i]
            
            if name == 'dust2':         # convert optical depth to mag
                data['params'][name] = {
                'samples': param_samples * 1.086,
                'map': MAP[name] * 1.086,   # Use the value from the best vector directly
                'q16': perc[name][0] * 1.086,
                'q50': perc[name][1] * 1.086,
                'q84': perc[name][2] * 1.086
            }
                
            else:
                data['params'][name] = {
                    'samples': param_samples,
                    'map': MAP[name],   # Use the value from the best vector directly
                    'q16': perc[name][0],
                    'q50': perc[name][1],
                    'q84': perc[name][2]
                }
    return data

phot_table = './Phot_Table_MIRI.fits'
with fits.open(phot_table) as hdul:
    galaxy_ids = hdul[1].data['ID']

prosp_dir = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.3/sourcephotonly_wMIRI/'

data = load_prospector_results(12717, prosp_dir)

# Bagpipes Loading Function

In [ ]:
bagp_dir = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/pipes/posterior/no_fesc_with_miri/'

example_id = 7102

def load_bagpipes_results(galaxy_id, bagp_dir):
    """Function to load Bagpipes result .h5 files and disect its 
    data structure to be used for easy plotting.

    Args:
        galaxy_id (int): The ID of the galaxy for which to load results
        bagp_dir (str): The directory containing the Bagpipes output files

    Returns:
        dict: A comprehensive dictionary storing the parameters samples, MAP values and quantiles
    """
    
    file = os.path.join(bagp_dir, f'{galaxy_id}.h5')    # Only one file per galaxy
    
    print(f"Loading Bagpipes output file: {file}")
    
    with h5py.File(file, 'r') as results:
        
        # Get redshift of the source
        fit_str = results.attrs['fit_instructions']
        fit = eval(fit_str, {"np": np, "array": np.array})
        zred = fit['redshift']
        
        # Extract the sampling results
        chain = results['samples2d']
        
        # Get maximum likelihood inde
        imax = np.argmax(results['lnlike'])
        
        # Manually extracted the labels from the results
        labels = ['dsfr1', 'dsfr2', 'dsfr3', 'dsfr4', 'dsfr5', 'dsfr6', 'logmass', 'logzsol', 'dust2', 
                    'duste_gamma', 'dust_index', 'duste_qpah', 'duste_umin', 'gas_logu']
        
        # Build the data structure
        data = {
            'meta': {'labels': labels, 'map_idx': imax, 'weights': None},
            'params': {}
        }
        
        # Loop through each parameter
        for i, name in enumerate(labels):
            samples = chain[:, i]
            q16, q50, q84 = quantile(samples, [0.16, 0.5, 0.84], weights=None)
            
            if name == 'logzsol':   # Convert metallicity to log
                data['params'][name] = {
                    'samples': np.log10(samples),
                    'map': np.log10(samples[imax]),
                    'q16': np.log10(q16),
                    'q50': np.log10(q50),
                    'q84': np.log10(q84)
                }
            
            else:    
                data['params'][name] = {
                    'samples': samples,
                    'map': samples[imax],
                    'q16': q16,
                    'q50': q50,
                    'q84': q84
                }

        data['params']['zred'] = {'samples': None, 'map': zred, 'q16': zred, 'q50': zred, 'q84': zred}
        
    return data
    
example_res = load_bagpipes_results(21424, bagp_dir)
print(example_res)


# Load results and store them in pickle files

In [ ]:
# Create a directory for your "Analysis Ready" data
output_dir = './comparison/processed_results'
os.makedirs(output_dir, exist_ok=True)

miri_table = "./Phot_Table_MIRI.fits"
id_table = Table.read(miri_table)
all_ids = [int(val.decode('utf-8') if isinstance(val, bytes) else val) 
                        for val in id_table['ID']]

print(len(all_ids))

no_spec = [9517, 9809, 11051, 11451, 12133, 17713, 17984, 20195, 20693, 20720, 21472, 22990]
bl_agn = [12020, 18977]
high_z = [11247, 7696]
intermediate_ap = [7136, 1904, 7922, 8469, 10314, 11337, 11420, 11451, 17517, 17669, 18332, 21452]

exclude = set(no_spec + bl_agn + intermediate_ap + high_z)

missing_from_table = [i for i in exclude if i not in set(all_ids)]

print(f"The following {len(missing_from_table)} IDs were in your 'exclude' list "
      f"but were NOT found in the MIRI table:\n{missing_from_table}")


print(len(exclude))
all_ids = [gal_id for gal_id in all_ids if gal_id not in exclude]

print(len(all_ids), "galaxies to process after exclusions.")

prosp_dir = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.3/sourcephotonly_wMIRI/'
bagp_dir = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/pipes/posterior/no_fesc_with_miri/'

for gal_id in all_ids:
    # 1. Load your existing functions
    try:
        p_data = load_prospector_results(gal_id, prosp_dir)
        b_data = load_bagpipes_results(gal_id, bagp_dir)
        
    except Exception as e:
        print(f"Error occurred while loading data for galaxy {gal_id}: {e}")
        continue

    # 2. Package them together
    combined_package = {
        'id': gal_id,
        'prospector': p_data,
        'bagpipes': b_data
    }

    # 3. Save to disk
    save_path = os.path.join(output_dir, f'{gal_id}_comp.pkl')
    with open(save_path, 'wb') as f:
        pkl.dump(combined_package, f)

# Plot the galaxies

In [ ]:
import corner
import matplotlib.pyplot as plt

def plot_corner(galaxy_id, b_data, p_data):
    # Select the labels you want to see
    # Using your existing labels: ['logmass', 'logzsol', 'dust2', 'gas_logu']
    plot_labels = ['logmass', 'logzsol', 'dust2', 'duste_gamma', 'dust_index', 'duste_qpah', 'duste_umin', 'gas_logu']
    
    # Extract the samples for these specific labels
    samples = np.array([b_data['params'][l]['samples'] for l in plot_labels]).T
    
    # Create the corner plot
    fig = corner.corner(
        samples,
        labels=[r"$\log_{10}(M_*)$", r"$\log_{10}(Z/Z_\odot)$", r"$A_V$", r"Dust $\gamma$", "Dust Index", r"Dust $q_{PAH}$", r"Dust $U_{min}$", r"Dust $U_{min}$", r"$\log_{10}(U)$"],
        quantiles=[0.16, 0.5, 0.84],
        weights=b_data['meta']['weights'],
        show_titles=True,
        title_kwargs={"fontsize": 12},
        color="Orange",
        smooth=1.0, # Helps visualize bimodality with only 500 samples
        # --- ADD THESE THREE LINES ---
        plot_datapoints=False,  # Suppresses the black dots
        fill_contours=True,     # Fills the 1/2/3 sigma levels with color
        plot_density=False      # Suppresses the grey background 'cloud'
    )
    
    # Extract the samples for these specific labels
    samples = np.array([p_data['params'][l]['samples'] for l in plot_labels]).T
    
    # Create the corner plot
    corner.corner(
        samples,
        fig=fig,  # Overlay on the same figure
        #labels=[r"$\log_{10}(M_*)$", r"$\log_{10}(Z/Z_\odot)$", r"$A_V$", r"Dust $\gamma$", "Dust Index", r"Dust $q_{PAH}$", r"Dust $U_{min}$", r"Dust $U_{min}$", r"$\log_{10}(U)$"],
        quantiles=[0.16, 0.5, 0.84],
        weights=p_data['meta']['weights'],
        show_titles=True,
        title_kwargs={"fontsize": 12},
        color="black",
        smooth=1.0, # Helps visualize bimodality with only 500 samples
        # --- ADD THESE THREE LINES ---
        plot_datapoints=False,  # Suppresses the black dots
        fill_contours=True,     # Fills the 1/2/3 sigma levels with color
        plot_density=False      # Suppresses the grey background 'cloud'
    )
    
    fig.suptitle(f"Galaxy {galaxy_id} - Bagpipes (Orange) vs Prospector (Black)", fontsize=24)
    plt.show()

b_data = load_bagpipes_results(21424, bagp_dir)
p_data = load_prospector_results(21424, prosp_dir)

plot_corner(21424, b_data, p_data)

# Look at a single Bagpipes output

In [ ]:
def plot_bagpipes_corner(galaxy_id, b_data):
    # Select the labels you want to see
    # Using your existing labels: ['logmass', 'logzsol', 'dust2', 'gas_logu']
    plot_labels = ['logmass', 'logzsol', 'dust2', 'duste_gamma', 'dust_index', 'duste_qpah', 'duste_umin', 'gas_logu']
    
    # Extract the samples for these specific labels
    samples = np.array([b_data['params'][l]['samples'] for l in plot_labels]).T
    
    # Create the corner plot
    fig = corner.corner(
        samples,
        labels=[r"$\log_{10}(M_*)$", r"$\log_{10}(Z/Z_\odot)$", r"$A_V$", r"Dust $\gamma$", "Dust Index", r"Dust $q_{PAH}$", r"Dust $U_{min}$", r"Dust $U_{min}$", r"$\log_{10}(U)$"],
        quantiles=[0.16, 0.5, 0.84],
        weights=b_data['meta']['weights'],
        show_titles=True,
        title_kwargs={"fontsize": 12},
        color="Orange",
        smooth=1.0, # Helps visualize bimodality with only 500 samples
        # --- ADD THESE THREE LINES ---
        plot_datapoints=False,  # Suppresses the black dots
        fill_contours=True,     # Fills the 1/2/3 sigma levels with color
        plot_density=False      # Suppresses the grey background 'cloud'
    )
    
    fig.suptitle(f"Galaxy {galaxy_id} Bagpipes", fontsize=16)
    plt.show()

example_res = load_bagpipes_results(21424, bagp_dir)

plot_bagpipes_corner(21424, example_res)

# Plot Prospector Corner Plot

In [ ]:
def plot_prospector_corner(galaxy_id, p_data):
    # Select the labels you want to see
    # Using your existing labels: ['logmass', 'logzsol', 'dust2', 'gas_logu']
    plot_labels = ['logmass', 'logzsol', 'dust2', 'duste_gamma', 'dust_index', 'duste_qpah', 'duste_umin', 'gas_logu']
    
    # Extract the samples for these specific labels
    samples = np.array([p_data['params'][l]['samples'] for l in plot_labels]).T
    
    # Create the corner plot
    fig = corner.corner(
        samples,
        labels=[r"$\log_{10}(M_*)$", r"$\log_{10}(Z/Z_\odot)$", r"$A_V$", r"Dust $\gamma$", "Dust Index", r"Dust $q_{PAH}$", r"Dust $U_{min}$", r"Dust $U_{min}$", r"$\log_{10}(U)$"],
        quantiles=[0.16, 0.5, 0.84],
        weights=p_data['meta']['weights'],
        show_titles=True,
        title_kwargs={"fontsize": 12},
        color="black",
        smooth=1.0, # Helps visualize bimodality with only 500 samples
        # --- ADD THESE THREE LINES ---
        plot_datapoints=False,  # Suppresses the black dots
        fill_contours=True,     # Fills the 1/2/3 sigma levels with color
        plot_density=False      # Suppresses the grey background 'cloud'
    )
    fig.suptitle(f"Galaxy {galaxy_id} Prospector", fontsize=16)
    plt.show()

example_res = load_prospector_results(21424, prosp_dir)

plot_prospector_corner(21424, example_res)